# Convergencia espacial — Variacion de N (numero de celdas)

Estudio de convergencia del modelo PSA 1D usando la **configuracion de referencia
identica a `test_steps_psa`**: misma columna (L=1.35 m, Di=0.30 m), isotermas DSL
ajustadas desde datos experimentales de Zeolite 13X, mismas condiciones de operacion
(P=9.5/9.0 bar, T=323 K, y=[0.33, 0.67]).

**Solo se varia N** — el numero de celdas del volumen finito:

| Eje | Valores |
|-----|---------|
| N (celdas) | 1, 5, 10, 20, 50, 100, 200|

**Metricas evaluadas:**
- Tiempo de computo wall-clock [s]
- Error de balance de masa [%]
- Convergencia L2 del perfil axial y_CO2(z) respecto a N=400 (referencia)
- Perfiles axiales al final del paso


## 1. Imports

In [6]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.interpolate import interp1d
warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.utils.isotherm_fitting import fit_single_T
from src.utils.isotherm_models  import arrh

from src.units.adsorber.config.gas_props  import build_gas_prop_config
from src.units.adsorber.config.adsorbent  import build_adsorbent_config
from src.units.adsorber.config.transport  import build_transport_config
from src.units.adsorber.config.thermal_bc import build_thermal_bc_config
from src.units.adsorber.config.boundary_c import build_boundary_c_config
from src.units.adsorber.config.initial_c  import build_initial_c_config
from src.units.adsorber.state             import pack_state_vector

from src.solvers.runner_adsorption import run_step
from src.postprocessing.adsorber_balances   import balance_report

DB_PATH   = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
DATA_PATH = os.path.join(os.getcwd(), "data", "Zeolite13X.csv")

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
print("Imports OK")
print("gasdb existe :", os.path.isfile(DB_PATH))
print("datos existen:", os.path.isfile(DATA_PATH))

Imports OK
gasdb existe : True
datos existen: True


## 2. Isotermas

**Identico a `test_steps_psa`**: ajuste DSL desde datos experimentales de Zeolite 13X
mediante `fit_single_T` + corrección de Arrhenius para dependencia con la temperatura.


In [7]:
df_iso = pd.read_csv(DATA_PATH)
P_CO2  = df_iso["Pressure_CO2 (bar)"].dropna().values
q_CO2  = df_iso["Uptake_CO2 (mol/kg)"].dropna().values
P_CH4  = df_iso["Pressure_CH4 (bar)"].dropna().values
q_CH4  = df_iso["Uptake_CH4 (mol/kg)"].dropna().values

T_REF  = 298.15    # K
dH_CO2 = 31.164e3  # J/mol
dH_CH4 =  9.856e3  # J/mol

iso_CO2_P, par_CO2, name_CO2, _ = fit_single_T(P_CO2, q_CO2)
iso_CH4_P, par_CH4, name_CH4, _ = fit_single_T(P_CH4, q_CH4)

def iso_CO2_PT(P_bar, T_K):
    return iso_CO2_P(np.asarray(P_bar) * arrh(float(T_K), dH_CO2, T_REF))

def iso_CH4_PT(P_bar, T_K):
    return iso_CH4_P(np.asarray(P_bar) * arrh(float(T_K), dH_CH4, T_REF))

def iso_fn(P_part_list, Ts):
    Ts_arr = np.asarray(Ts, dtype=float)
    N_loc  = len(Ts_arr)
    q_eq   = np.zeros((2, N_loc), dtype=float)
    for n in range(N_loc):
        q_eq[0, n] = iso_CO2_PT(float(P_part_list[0][n]), Ts_arr[n])
        q_eq[1, n] = iso_CH4_PT(float(P_part_list[1][n]), Ts_arr[n])
    return q_eq

print(f"Isoterma CO2: {name_CO2}  |  CH4: {name_CH4}")


Isoterma CO2: Dual-site Langmuir  |  CH4: Dual-site Langmuir


## 3. Parametros del sistema

**Identico a `test_steps_psa`**. Solo N variara en el benchmark.


In [8]:
# Geometria — identica a test_steps_psa
nc     = 2
SPECIES = ["CO2", "CH4"]
L      = 1.35          # m
Di     = 0.30          # m
Do     = 0.35          # m
Ai     = 0.25 * np.pi * Di**2
Pi     = np.pi * Di
Po     = np.pi * Do
e_wall = (Do - Di) / 2.0

# Adsorbente — identico a test_steps_psa
epsi       = 0.37
D_particle = 12.0e-4    # m
rho_s      = 3200.0     # kg/m3
Cp_s       = 1200.0     # J/kg/K
k_s        = 10.0       # W/m/K
tau_pore   = 3.0
r_pore     = 1.0e-9     # m
dH_ads     = np.array([dH_CO2, dH_CH4])

# Condiciones de operacion — identicas a test_steps_psa
P_inlet  = 9.5          # bar
P_outlet = 9.0          # bar
T_feed   = 323.0        # K
y_feed   = np.array([0.33, 0.67])
Q_feed   = 0.05 * Ai    # m3/s
Cv_in    = 2.5e-1
Cv_out   = 1.5

# Parametros del integrador
N_REF  = 200            # referencia para L2 (mayor resolucion)
N_VALUES = [1, 5, 10, 20, 50, 100, 200]
T_MAX  = 600.0           # s — paso de adsorcion
N_SEC  = 20             # pts/s — resolucion temporal para balance sub-1%
RTOL   = 1.0e-8
ATOL   = 1.0e-10
SOLVER = "solve_ivp"

print(f"Sistema : L={L} m  Di={Di} m  N_VALUES={N_VALUES}")
print(f"Operacion: P={P_inlet}/{P_outlet} bar  T={T_feed} K  T_MAX={T_MAX} s")


Sistema : L=1.35 m  Di=0.3 m  N_VALUES=[1, 5, 10, 20, 50, 100, 200]
Operacion: P=9.5/9.0 bar  T=323.0 K  T_MAX=600.0 s


## 4. Funcion `build_params(N_nodes)`

Construye `(params, sv0)` con la configuracion de referencia para cualquier N.
Replica exactamente el setup de `test_steps_psa` con N como unico parametro libre.


In [9]:
def build_params(N_nodes: int):
    """
    Construye (params, sv0) para N_nodes celdas con la configuracion
    identica a test_steps_psa (mismo sistema fisico, mismas BCs).

    Parameters
    ----------
    N_nodes : int — numero de celdas del volumen finito

    Returns
    -------
    params : dict — listo para run_step()
    sv0    : ndarray — vector de estado inicial
    """
    dz = L / N_nodes

    # 1. Propiedades del gas
    prop_gas = build_gas_prop_config(
        species=SPECIES, mode="constant", n_comp=nc, N=N_nodes,
        T_ref=T_feed, db_path=DB_PATH,
    )
    gas_T_ref = float(np.min(np.asarray(prop_gas["Tref"])))

    # 2. Adsorbente + isoterma
    ads = build_adsorbent_config(
        iso_fn=iso_fn, n_comp=nc, N=N_nodes,
        epsi=epsi, D_particle=D_particle,
        rho_s=rho_s, Cp_solid=Cp_s, k_solid=k_s,
        dH_adsorption=dH_ads, tau=tau_pore, r_pore=r_pore,
    )

    # 3. Transporte (correlaciones)
    trans = build_transport_config(mode="correlation", n_comp=nc, N=N_nodes)

    # 4. Pared (ambient_htc, identico a test_steps_psa)
    thermal_bc = build_thermal_bc_config(
        mode="ambient_htc", Di=Di, Do=Do, e_wall=e_wall,
        h_ambi=250.0, T_ambi=298.0, k_wall=2.0,
    )

    # 5. Contornos
    bc = build_boundary_c_config(
        n_comp=nc, N=N_nodes, forward_flow_direction=True,
        Q_in_feed=Q_feed,  P_in_feed=P_inlet,  T_in_feed=T_feed,  y_in_feed=y_feed,
        P_out_ads=P_outlet, Cv_out_ads=Cv_out,
        Q_in_purge=Q_feed, P_in_purge=P_inlet, T_in_purge=T_feed, y_in_purge=y_feed,
        P_out_purge=P_outlet, Cv_out_purge=Cv_out,
        P_high=P_inlet, Cv_in_prfeed=Cv_in,
        P_low=1.0, Cv_out_blowdown=Cv_out, Cv_eq_lm=Cv_out, Cv_eq_mh=Cv_out,
    )

    # 6. Condiciones iniciales (columna casi limpia, 1% CO2)
    y_init = np.array([0.01 * np.ones(N_nodes), 0.99 * np.ones(N_nodes)])
    P_init = P_inlet * np.ones(N_nodes)
    T_init = T_feed  * np.ones(N_nodes)
    ic = build_initial_c_config(
        P_init=P_init, Tg_init=T_init, Ts_init=T_init, y_init=y_init,
        n_comp=nc, N=N_nodes, iso_fn=ads["iso"],
        prop_gas=prop_gas, epsi=epsi, gas_T_ref=gas_T_ref, q_init=None,
    )
    sv0 = pack_state_vector(
        C=ic["C_init"], q=ic["q_init"],
        Hg=ic["Hg_init"], Ts=ic["Ts_init"],
    )

    params = {
        "n_comp": nc, "N": N_nodes, "dz": dz,
        "Ai": Ai, "Di": Di, "Pi": Pi, "Po": Po,
        "prop_gas":          prop_gas,
        "MW":                np.asarray(prop_gas["MW"]),
        "gas_T_ref":         gas_T_ref,
        "iso_fn":            ads["iso"],
        "epsi":              ads["epsi"],
        "rho_s":             ads["rho_s"],
        "Cp_s":              ads["Cp_s"],
        "k_s":               ads["k_s"],
        "dH":                ads["dH_comp"],
        "prop_lecho":        ads["prop_lecho"],
        "bc_config":         bc,
        "trans_config":      trans,
        "thermal_bc_config": thermal_bc,
        "prop_update_mode":  "frozen",
        "trans_update_mode": "frozen",
        "energy":            True,
        "Tg_init":           T_init.copy(),
        "species":           SPECIES,
        "_cache":            {},
    }
    return params, sv0

# Smoke test
p_test, sv_test = build_params(21)
print(f"build_params OK | N=21 | sv_size={sv_test.size}")


build_params OK | N=21 | sv_size=126


## 5. Bucle de benchmark — N = [1, 5, 10, 20, 50, 100, 200]

Para cada N: `build_params(N)` → `run_step()` → `balance_report()`.


In [ ]:
RESULTS = {}    # {N: dict con col, params, metricas}
n_total = len(N_VALUES)

for idx, N in enumerate(N_VALUES):
    print(f"[{idx+1:02d}/{n_total}] N={N:4d} ...", end=" ", flush=True)
    try:
        params, sv0 = build_params(N)

        t0 = time.perf_counter()
        t_arr, y_hist, col = run_step(
            "ads", sv0, T_MAX, params,
            solver=SOLVER, rtol=RTOL, atol=ATOL,
            n_sec=N_SEC, show_progress=False,
        )
        wall_time = time.perf_counter() - t0

        df_bal = balance_report(col, params, include_energy=False, print_report=False)

        def _get(df, row, col_name="Error rel [%]"):
            try: return float(df.loc[row, col_name])
            except: return float("nan")

        mass_err = abs(_get(df_bal, "N_total [mol]"))

        RESULTS[N] = {
            "col": col, "params": params, "t_arr": t_arr,
            "wall_time": wall_time, "mass_err_pct": mass_err,
            "sv_size": sv0.size, "ok": True,
        }
        print(f"t={wall_time:.2f} s  |  mass_err={mass_err:.3f}%")

    except Exception as exc:
        RESULTS[N] = {"ok": False, "error": str(exc)}
        print(f"FAILED: {str(exc)[:80]}")

print("\nBucle completado.")


[01/7] N=   1 ... t=3.74 s  |  mass_err=0.000%
[02/7] N=   5 ... t=8.95 s  |  mass_err=0.000%
[03/7] N=  10 ... t=24.91 s  |  mass_err=0.000%
[04/7] N=  20 ... t=117.16 s  |  mass_err=0.000%
[05/7] N=  50 ... t=1051.17 s  |  mass_err=0.000%
[06/7] N= 100 ... t=6838.08 s  |  mass_err=0.000%
[07/7] N= 200 ... 

## 6. Dashboard A — Tiempo de computo y balance de masa

In [ ]:
Nv  = [N for N in N_VALUES if RESULTS[N].get("ok")]
wt  = [RESULTS[N]["wall_time"]    for N in Nv]
mb  = [RESULTS[N]["mass_err_pct"] for N in Nv]
sv  = [RESULTS[N]["sv_size"]      for N in Nv]

rows = []
for N in Nv:
    r = RESULTS[N]
    rows.append({"N": N, "sv_size": r["sv_size"],
                 "wall_time [s]": round(r["wall_time"], 3),
                 "mass_err [%]":  round(r["mass_err_pct"], 4)})
df_sum = pd.DataFrame(rows).set_index("N")
display(df_sum.style
        .background_gradient(cmap="YlOrRd", subset=["wall_time [s]"])
        .background_gradient(cmap="RdYlGn_r", subset=["mass_err [%]"])
        .format(precision=4))

N_arr = np.array(Nv, dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Convergencia espacial — Tiempo y balance de masa", fontsize=11, fontweight="bold")

ax = axes[0]
ax.loglog(Nv, wt, "o-", color="steelblue", lw=2, label="Medido")
if len(wt) > 1:
    ax.loglog(N_arr, wt[0]*(N_arr/N_arr[0])**1.5, "k--", alpha=0.5, label="O(N^1.5)")
    ax.loglog(N_arr, wt[0]*(N_arr/N_arr[0])**2,   "k:",  alpha=0.4, label="O(N^2)")
ax.set_xlabel("N (celdas)"); ax.set_ylabel("Wall time [s]")
ax.set_title("Tiempo de computo vs N")
ax.legend(); ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
mb_valid = [m for m in mb if np.isfinite(m) and m > 0]
if mb_valid:
    ax.semilogy(Nv, mb, "s-", color="crimson", lw=2)
    ax.axhline(1.0,  color="orange", ls="--", lw=1.2, label="1%")
    ax.axhline(0.1,  color="green",  ls="--", lw=1.2, label="0.1%")
else:
    ax.plot(Nv, mb, "s-", color="crimson", lw=2)
ax.set_xlabel("N (celdas)"); ax.set_ylabel("|Error balance masa| [%]")
ax.set_title("Balance de masa vs N")
ax.legend(); ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Perfiles axiales al final del paso — efecto de N

In [ ]:
colors = cm.viridis(np.linspace(0.1, 0.9, len(Nv)))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f"Perfiles axiales al final del paso (t = {T_MAX} s)", fontsize=11, fontweight="bold")

vars_info = [
    ("_y_results",  0,    "y_CO2 [-]",  "Fraccion molar CO2"),
    ("_y_results",  1,    "y_CH4 [-]",  "Fraccion molar CH4"),
    ("_Tg_results", None, "Tg [K]",     "Temperatura del gas"),
    ("_P_results",  None, "P [bar]",    "Presion"),
]

for ax, (attr, idx, ylabel, title) in zip(axes.flat, vars_info):
    for i, N in enumerate(Nv):
        col = RESULTS[N]["col"]
        z   = col._z
        data = getattr(col, attr)
        if idx is not None:
            y_data = data[-1, idx, :]
        else:
            y_data = data[-1, :]
        ax.plot(z, y_data, color=colors[i], lw=1.5, label=f"N={N}")
    ax.set_xlabel("z [m]"); ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Convergencia L2 — referencia N=200

Error L2 relativo del perfil `y_CO2(z)` y `Tg(z)` respecto a la solucion con N=200.
Indica el orden de convergencia espacial del esquema de volumen finito.


In [ ]:
N_REF_KEY = N_REF  # 100

if not RESULTS.get(N_REF_KEY, {}).get("ok"):
    print(f"Referencia N={N_REF_KEY} no disponible — no se puede calcular L2")
else:
    col_ref = RESULTS[N_REF_KEY]["col"]
    z_ref   = col_ref._z

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"Convergencia L2 — referencia N={N_REF_KEY}", fontsize=11, fontweight="bold")

    for panel, (attr, idx, var_label) in enumerate([
        ("_y_results",  0,    "y_CO2"),
        ("_Tg_results", None, "Tg [K]"),
    ]):
        ax = axes[panel]
        f_ref = col_ref._y_results[-1, 0, :] if attr == "_y_results" else col_ref._Tg_results[-1, :]
        interp_ref = interp1d(z_ref, f_ref, kind="linear", fill_value="extrapolate")

        Nv_L2, errs_L2 = [], []
        for N in Nv:
            if N == N_REF_KEY or not RESULTS[N].get("ok"):
                continue
            col_N = RESULTS[N]["col"]
            z_N   = col_N._z
            f_N   = col_N._y_results[-1, 0, :] if attr == "_y_results" else col_N._Tg_results[-1, :]
            f0    = interp_ref(z_N)
            norm0 = np.linalg.norm(f0)
            err   = np.linalg.norm(f_N - f0) / max(norm0, 1e-20)
            Nv_L2.append(N); errs_L2.append(err)

        if Nv_L2:
            ax.loglog(Nv_L2, errs_L2, "o-", color="steelblue", lw=2, label="Error L2")
            N_line = np.array([min(Nv_L2), max(Nv_L2)], dtype=float)
            e0 = errs_L2[0]
            ax.loglog(N_line, e0*(N_line/N_line[0])**(-1), "k--", alpha=0.5, label="O(1/N)")
            ax.loglog(N_line, e0*(N_line/N_line[0])**(-2), "k:",  alpha=0.4, label="O(1/N^2)")

        ax.set_xlabel("N (celdas)"); ax.set_ylabel("Error L2 relativo [-]")
        ax.set_title(f"Convergencia L2 — {var_label}")
        ax.legend(); ax.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()


## 9. Series temporales — fraccion molar CO2 a la salida (z = L)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle("y_CO2 en la salida (z = L) — efecto de N", fontsize=11, fontweight="bold")

for i, N in enumerate(Nv):
    col   = RESULTS[N]["col"]
    t_arr = RESULTS[N]["t_arr"]
    ax.plot(t_arr, col._y_results[:, 0, -1], color=colors[i], lw=1.4, label=f"N={N}")

ax.set_xlabel("t [s]"); ax.set_ylabel("y_CO2 salida [-]")
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Tabla resumen

In [ ]:
print("=" * 60)
print(f"  CONVERGENCIA ESPACIAL — {len(Nv)} casos completados")
print(f"  Config: L={L}m  Di={Di}m  P={P_inlet}/{P_outlet}bar  T={T_feed}K")
print(f"  T_MAX={T_MAX}s  N_SEC={N_SEC}  RTOL={RTOL}  ATOL={ATOL}")
print("=" * 60)
print(f"{'N':>6}  {'sv_size':>8}  {'wall_time [s]':>14}  {'mass_err [%]':>13}  {'calidad':>10}")
print("-" * 60)
for N in Nv:
    r = RESULTS[N]
    err = r["mass_err_pct"]
    if err < 0.1:   label = "EXCELENTE"
    elif err < 1.0: label = "BUENO"
    elif err < 5.0: label = "ACEPTABLE"
    else:           label = "REVISAR"
    print(f"{N:>6}  {r['sv_size']:>8}  {r['wall_time']:>14.3f}  {err:>13.4f}  {label:>10}")
print("=" * 60)
